# 第01章《误差的代价》C2 人工复核

G19 · 首轮交叉审查 · 2026-11-21

本 Notebook 用最小输入核验浮点中点、差分取样、条件性与交叉熵的边界。推导与材料位置见 [手动推导](C2组-手动推导.md) 和 [批判报告](C2组-批判报告.md)。代码由 Codex 根据独立推导编写，并用 Jupyter 内核执行。运行仅需 Python 标准库。G19 组员已复核本 Notebook。

In [1]:
from decimal import Decimal, localcontext
import math
import platform
import sys

assert sys.float_info.radix == 2 and sys.float_info.mant_dig == 53
print(f"Python {platform.python_version()} | {platform.system()} {platform.machine()} | binary64")
print("固定输入，无随机数。每个高精度参考使用独立的局部 Decimal 上下文。")

Python 3.12.10 | Darwin arm64 | binary64
固定输入，无随机数。每个高精度参考使用独立的局部 Decimal 上下文。


## 1.1、1.2　同一个零可以来自不同的舍入步骤

关联 CH01-G19-C002。在 $10^{16}$ 处，$x+1$ 舍回 $x$。在 $2^{52}$ 处，$x+1$ 可准确表示，但两个平方根舍为同值。用 80 位和 100 位 Decimal 核对参考收敛。有理化式的相对容差取 $10^{-15}$，为 binary64 的少量运算留出舍入余量。中点、相等性及零输出用精确断言。

In [2]:
def sqrt_difference_reference(x, precision):
    with localcontext() as ctx:
        ctx.prec = precision
        x_exact = Decimal.from_float(x)
        return 1 / ((x_exact + 1).sqrt() + x_exact.sqrt())


for x, expected_increment in [(1e16, 0.0), (float(2**52), 1.0)]:
    root = math.sqrt(x)
    root_next = math.sqrt(x + 1.0)
    direct = root_next - root
    rationalized = 1.0 / (root_next + root)
    ref80 = sqrt_difference_reference(x, 80)
    ref100 = sqrt_difference_reference(x, 100)
    with localcontext() as ctx:
        ctx.prec = 100
        ref_change = abs((ref80 - ref100) / ref100)
        rel_error = abs((Decimal.from_float(rationalized) - ref100) / ref100)
    assert ref_change < Decimal("1e-78")
    assert (x + 1.0) - x == expected_increment
    assert direct == 0.0 and rel_error < Decimal("1e-15")
    print(f"x={x:.16g} | ULP={math.ulp(x):g} | 所存增量={expected_increment:g}")
    print(f"  直接式={direct:g} | 有理化={rationalized:.17g} | 相对误差={rel_error:.3E}")

assert math.ulp(1e8) == 2.0**-26
print(f"binary64 在 1e8 处的间距 = 2^-26 = {math.ulp(1e8):.17g}")

x=1e+16 | ULP=2 | 所存增量=0
  直接式=0 | 有理化=5.0000000000000001e-09 | 相对误差=4.592E-17
x=4503599627370496 | ULP=1 | 所存增量=1
  直接式=0 | 有理化=7.4505805969238281e-09 | 相对误差=5.551E-17
binary64 在 1e8 处的间距 = 2^-26 = 1.4901161193847656e-08


## 1.2　区分小步长的截断误差与实现误差

关联 CH01-G19-C002。对 $e^x$ 在 $x=1$ 处比较直接差分与 $e\,\operatorname{expm1}(h)/h$。高精度差商与解析导数分别衡量实现和截断误差。改写依赖本例的指数结构。断言核对取样点重合、截断项的符号和改写相对准确度。

In [3]:
h = 1e-16
direct = (math.exp(1.0 + h) - math.exp(1.0)) / h
reformulated = math.e * math.expm1(h) / h
with localcontext() as ctx:
    ctx.prec = 90
    h_exact = Decimal.from_float(h)
    derivative_ref = Decimal(1).exp()
    quotient_ref = derivative_ref * (h_exact.exp() - 1) / h_exact
    truncation = quotient_ref - derivative_ref
    reformulated_rel_error = abs((Decimal.from_float(reformulated) - quotient_ref) / quotient_ref)
assert 1.0 + h == 1.0 and direct == 0.0
assert Decimal("1e-16") < truncation < Decimal("2e-16")
assert reformulated_rel_error < Decimal("1e-15")
print(f"h={h:.1e} | 所存取样点增量={(1.0 + h) - 1.0:g}")
print(f"直接差分={direct:g} | 改写差分={reformulated:.17g}")
print(f"高精度截断误差={truncation:.6E} | 改写相对参考差商误差={reformulated_rel_error:.3E}")

h=1.0e-16 | 所存取样点增量=0
直接差分=0 | 改写差分=2.7182818284590451
高精度截断误差=1.359141E-16 | 改写相对参考差商误差=1.032E-16


## 1.3　局部条件数与有限扰动

关联 CH01-G19-C003。对 $q(x)=1/(1-x)$，在 $x=0.9999$ 的局部条件数为 9999。有限扰动放大率随扰动逼近奇点而升高。对 D3 的开方例，按 $a\mapsto\sqrt a$ 定义输入，用高精度有限扰动对照解析条件数 $1/2$。D3 原式的分母是计算根的残差，可能舍为零。

In [4]:
with localcontext() as ctx:
    ctx.prec = 80
    x = Decimal("0.9999")
    kappa_q = abs(x / (1 - x))
    assert kappa_q == 9999
    for delta, expected in [(Decimal("1e-8"), Decimal(10000)),
                             (Decimal("0.000099"), Decimal(999900))]:
        q0 = 1 / (1 - x)
        q1 = 1 / (1 - x - delta)
        amplification = abs((q1 - q0) / q0) / abs(delta / x)
        assert abs(amplification - expected) < Decimal("1e-70")
        print(f"q: d={delta:g}, 有限扰动放大率={amplification:.8f}, 局部条件数={kappa_q}")
    assert x + Decimal("0.0001") == 1
    a, perturbation = Decimal(2), Decimal("1e-8")
    root = a.sqrt()
    ratio = (((a + perturbation).sqrt() - root) / root) / (perturbation / a)
    assert abs(ratio - Decimal("0.5")) < Decimal("1e-8")

x_stored = math.sqrt(2.0)
residual = x_stored * x_stored - 2.0
original_formula = abs(2 * x_stored) * 2.0 / abs(residual) if residual else math.inf
assert original_formula > 1e12
print(f"sqrt: 有限扰动放大率={ratio:.12f} | 解析相对条件数=0.5")
print(f"D3 残差={residual:.6e} | D3 原式={original_formula:.6e}")

q: d=1e-8, 有限扰动放大率=10000.00000000, 局部条件数=9999
q: d=0.000099, 有限扰动放大率=999900.00000000, 局部条件数=9999
sqrt: 有限扰动放大率=0.499999999375 | 解析相对条件数=0.5
D3 残差=4.440892e-16 | D3 原式=1.273810e+16


## 1.4　有限值和归一化通过时，损失与梯度仍可能丢失

关联 CH01-G19-C004。输入为 binary64 可精确表示的 logits，下标从 0 开始。80 位 Decimal 直接计算参考损失和梯度。对照实现将一个最大项从指数和中取出，用 log1p 保留小修正，用其他类概率之和计算目标类梯度。对非零结果用相对容差 $10^{-14}$，保留低分目标与并列最大对照。

In [5]:
def ce_comparison(logits, target):
    peak_index = max(range(len(logits)), key=logits.__getitem__)
    peak = logits[peak_index]
    weights = [math.exp(z - peak) for z in logits]
    total = sum(weights)
    probabilities = [weight / total for weight in weights]

    loss = (peak - logits[target]) + math.log(total)
    target_grad = probabilities[target] - 1
    tail = math.fsum(weight for index, weight in enumerate(weights) if index != peak_index)
    revised_loss = (peak - logits[target]) + math.log1p(tail)
    revised_grad = -math.fsum(p for index, p in enumerate(probabilities) if index != target)

    with localcontext() as ctx:
        ctx.prec = 80
        exact_weights = [(Decimal(z) - Decimal(peak)).exp() for z in logits]
        exact_total = sum(exact_weights)
        ref_loss = Decimal(peak - logits[target]) + exact_total.ln()
        ref_grad = exact_weights[target] / exact_total - 1
        loss_error = abs((Decimal.from_float(revised_loss) - ref_loss) / ref_loss)
        grad_error = abs((Decimal.from_float(revised_grad) - ref_grad) / ref_grad)
    assert all(math.isfinite(p) for p in probabilities)
    assert math.isclose(sum(probabilities), 1, rel_tol=1e-15)
    assert loss_error < Decimal("1e-14") and grad_error < Decimal("1e-14")
    print(f"z={logits}, y={target}, sum(p)={sum(probabilities):.17g}")
    print(f"  损失原式={loss:.17g}, 参考={ref_loss:.15E}, 改写={revised_loss:.17g}")
    print(f"  目标梯度原式={target_grad:.17g}, 参考={ref_grad:.15E}, 改写={revised_grad:.17g}")
    print(f"  改写相对误差：loss={loss_error:.3E}, target gradient={grad_error:.3E}")
    return loss, target_grad


assert ce_comparison([1000, 960], 0) == (0.0, 0.0)
assert ce_comparison([1000, 0, -1000], 2) == (2000.0, -1.0)
ce_comparison([1000, 1000], 0)

z=[1000, 960], y=0, sum(p)=1
  损失原式=0, 参考=4.248354255291589E-18, 改写=4.2483542552915889e-18
  目标梯度原式=0, 参考=-4.248354255291589E-18, 改写=-4.2483542552915889e-18
  改写相对误差：loss=2.715E-17, target gradient=2.503E-17
z=[1000, 0, -1000], y=2, sum(p)=1
  损失原式=2000, 参考=2.000000000000000E+3, 改写=2000
  目标梯度原式=-1, 参考=-1.000000000000000E+0, 改写=-1
  改写相对误差：loss=0.000E+3, target gradient=0.000E+3
z=[1000, 1000], y=0, sum(p)=1
  损失原式=0.69314718055994529, 参考=6.931471805599453E-1, 改写=0.69314718055994529
  目标梯度原式=-0.5, 参考=-5.000000000000000E-1, 改写=-0.5
  改写相对误差：loss=3.346E-17, target gradient=0.000E+3


(0.6931471805599453, -0.5)

## 1.4　先组成绝对尺度的 LSE，也可能丢失交叉熵

关联 CH01-G19-C005。复写 B1-42 的先算 LSE 再减目标值的运算顺序。用两个相等且精确可表示的 logits，准确概率都是 $1/2$，损失为 $\log 2$。对照方案把两个大数之差提前。

In [6]:
z = [1e16, 1e16]
peak = max(z)
log_shifted_sum = math.log(sum(math.exp(value - peak) for value in z))
b1_path = (peak + log_shifted_sum) - z[-1]
a_path = (peak - z[-1]) + log_shifted_sum
with localcontext() as ctx:
    ctx.prec = 80
    ref = Decimal(2).ln()
    rel_error = abs((Decimal.from_float(a_path) - ref) / ref)
assert b1_path == 0 and rel_error < Decimal("1e-15")
print(f"ULP={math.ulp(peak):g} | B1 运算顺序={b1_path:g} | A 稿运算顺序={a_path:.17g}")
print(f"80 位参考={ref:.17E} | A 稿运算顺序相对误差={rel_error:.3E}")
print("全部最小核验通过。")

ULP=2 | B1 运算顺序=0 | A 稿运算顺序=0.69314718055994529
80 位参考=6.93147180559945309E-1 | A 稿运算顺序相对误差=3.346E-17
全部最小核验通过。


## 原稿运行记录

原稿运行在独立的临时工作副本中，清除历史输出后使用新的 Jupyter 内核。运行环境为 Python 3.12.10、NumPy 1.26.4、Matplotlib 3.9.2，macOS arm64／CPU。本页上方为 C2 独立代码。

| 对象 | 已执行代码格 | 本次观察 |
| --- | ---: | --- |
| A1 公开实验原样 | 1／15 | 第 2 格（environment）找不到代码要求的 A1组-正文.md，抛出 RuntimeError |
| A1 临时加正文别名 | 15／15 | 所有格运行完成。别名取自同目录公开更新正文，源码未改 |
| A2 公开实验 | 14／14 | 所有格运行完成 |
| B1 验证本 | 5／5 | 所有格运行完成 |
| D3 整合实验 | 5／5 | 所有格运行完成。第 7 格的错误条件数仍会输出 |
| B2 | — | 其 PR 和 main 未提供验证 Notebook，已阅读验证报告与修正清单 |

源版本与单元定位见 [批判报告](C2组-批判报告.md)。以上为 Codex 实际执行记录。